<a href="https://colab.research.google.com/github/clee2026/MSDS_498_Capstone/blob/main/init_findings/notebook_1_Understand_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/clee2026/MSDS_498/blob/main/capstone/findings/notebook_1_intake_profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1 - Intake and Profiling for Initial Findings

This notebook is the first stage of the project pipeline.

## Purpose
This notebook does intake and profiling only. It reads the raw parquet dataset from Google Drive / MyDrive, profiles the data, saves the profiling outputs to one local Colab output directory, and creates the handoff files that will drive Notebook 2.

## What this notebook covers in the deliverable
This notebook supports these report sections:

- **Description of Data**
- **Overview of the Data**

It does **not** do transformation or modeling.

## Main outputs
This notebook will create these core files inside /content/project_outputs/01_intake_profile/:

- dataset_inventory.csv
- column_profile.csv
- missingness_summary.csv
- distinct_value_summary.csv
- candidate_drop_review.csv
- top_values_selected_columns.csv
- initial_findings_notes.txt
- run_log.txt

It will also save selected charts and a run_manifest.json file in /content/project_outputs/.

In [ ]:
# 0. Optional installs
# In most Colab environments, pandas, numpy, matplotlib, and pyarrow
# are already installed. If external to Colab, see below pip install

# !pip install pyarrow pandas matplotlib numpy

In [ ]:
# 1. Imports

import os
import gc
import json
import math
import time
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive

import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)

In [ ]:
# 2. User configuration

# SOURCE_PARQUET = "/content/drive/MyDrive/Capstone/nyc311/311_raw.parquet"

# SOURCE_PARQUET = "/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/nyc_311_all_appended.parquet"

SOURCE_PARQUET = "/content/drive/MyDrive/project_data/Capstone Course Project Files/Data/nyc_311_FINAL_UPDATED.parquet"

# Local Colab output directory for all downloadable artifacts.
OUTPUT_DIR = "/content/project_outputs"

# Main notebook output folder.
PROFILE_DIR = os.path.join(OUTPUT_DIR, "01_intake_profile")
PLOT_DIR = os.path.join(PROFILE_DIR, "plots")

# Sampling and profiling controls.
# These defaults are tuned for large parquet files in Colab with limited RAM.
RANDOM_SEED = 42
SAMPLE_ROWS_TARGET = 150000          # development sample for charts and sample-based profiling
SAMPLE_BATCH_SIZE = 25000            # rows per Arrow batch when creating the sample
PROFILE_SERIES_PLOT_MAX = 50000      # max rows used in a single chart
MAX_TOP_VALUES_PER_COLUMN = 15       # how many top categories to save for selected columns
MAX_SAMPLE_VALUES_PER_COLUMN = 5     # sample example values stored in column_profile
MAX_TEXT_LENGTH = 120                # truncate long sample strings for readability

# Deep profile candidates are selected automatically from the schema.
MAX_NUMERIC_PLOT_COLUMNS = 8
MAX_CATEGORICAL_PLOT_COLUMNS = 6
MAX_DATETIME_PLOT_COLUMNS = 4

# If True, the notebook will scan the full dataset in Arrow batches
# to compute full null counts by column. This uses Arrow null counts
# directly and avoids converting every batch to pandas.
RUN_FULL_NULL_SCAN = True

# Number of rows to process per batch during the full null scan.
# Lower this if RAM becomes tight.
NULL_SCAN_BATCH_SIZE = 50000

# If True, row count will be taken from parquet metadata.
USE_METADATA_ROW_COUNT = True


In [ ]:
# 3. Mount Google Drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 4. Create output folders

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROFILE_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("PROFILE_DIR:", PROFILE_DIR)
print("PLOT_DIR:", PLOT_DIR)

OUTPUT_DIR: /content/project_outputs
PROFILE_DIR: /content/project_outputs/01_intake_profile
PLOT_DIR: /content/project_outputs/01_intake_profile/plots


In [ ]:
# 5. Validate source path and start run log

if not os.path.exists(SOURCE_PARQUET):
    raise FileNotFoundError(
        f"Could not find SOURCE_PARQUET:\n{SOURCE_PARQUET}\n\n"
        "Update the SOURCE_PARQUET variable in Section 2 and rerun this cell."
    )

RUN_TS = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

run_log_path = os.path.join(PROFILE_DIR, "run_log.txt")

with open(run_log_path, "w") as f:
    f.write("Notebook 1 - Intake and Profiling\n")
    f.write(f"Run timestamp: {RUN_TS}\n")
    f.write(f"Run ID: {RUN_ID}\n")
    f.write(f"Source parquet: {SOURCE_PARQUET}\n")
    f.write(f"Output dir: {OUTPUT_DIR}\n")

print("Run log initialized:", run_log_path)

Run log initialized: /content/project_outputs/01_intake_profile/run_log.txt


## Helper functions


In [ ]:
# 6. Helper functions

def append_run_log(message: str, path: str = run_log_path) -> None:
    with open(path, "a") as f:
        f.write(message.rstrip() + "\n")

def safe_string(value, max_len=MAX_TEXT_LENGTH):
    if pd.isna(value):
        return None
    text = str(value)
    if len(text) > max_len:
        text = text[:max_len] + "..."
    return text

def infer_semantic_type(arrow_type: pa.DataType) -> str:
    if pa.types.is_integer(arrow_type) or pa.types.is_floating(arrow_type) or pa.types.is_decimal(arrow_type):
        return "numeric"
    if pa.types.is_boolean(arrow_type):
        return "boolean"
    if pa.types.is_date(arrow_type) or pa.types.is_timestamp(arrow_type):
        return "datetime"
    if pa.types.is_string(arrow_type) or pa.types.is_large_string(arrow_type):
        return "string"
    if pa.types.is_binary(arrow_type) or pa.types.is_large_binary(arrow_type):
        return "binary"
    if pa.types.is_list(arrow_type) or pa.types.is_large_list(arrow_type):
        return "list"
    if pa.types.is_struct(arrow_type):
        return "struct"
    return "other"

def choose_plot_columns(column_profile_df: pd.DataFrame):
    numeric_cols = column_profile_df.loc[column_profile_df["semantic_type"] == "numeric", "column_name"].tolist()
    categorical_cols = column_profile_df.loc[
        column_profile_df["semantic_type"].isin(["string", "boolean"]), "column_name"
    ].tolist()
    datetime_cols = column_profile_df.loc[column_profile_df["semantic_type"] == "datetime", "column_name"].tolist()

    numeric_cols = numeric_cols[:MAX_NUMERIC_PLOT_COLUMNS]
    categorical_cols = categorical_cols[:MAX_CATEGORICAL_PLOT_COLUMNS]
    datetime_cols = datetime_cols[:MAX_DATETIME_PLOT_COLUMNS]
    return numeric_cols, categorical_cols, datetime_cols

def classify_recommended_action(null_pct, distinct_estimate, semantic_type, sample_non_null_count, column_name):
    name_lower = str(column_name).lower()

    if null_pct >= 95:
        return "drop", "Very high missingness"
    if sample_non_null_count == 0:
        return "drop", "No non-null values observed"
    if distinct_estimate == 1 and semantic_type not in ["list", "struct", "binary"]:
        return "drop", "Appears constant in sample"
    if any(token in name_lower for token in ["ssn", "social_security", "taxpayer", "passport"]):
        return "review", "Potential sensitive field"
    if any(token in name_lower for token in ["id", "key", "guid", "uuid"]) and semantic_type in ["string", "numeric"]:
        return "review", "Possible identifier field"
    if semantic_type in ["list", "struct", "binary"]:
        return "review", "Complex type should be reviewed"
    if semantic_type == "string" and distinct_estimate > 10000:
        return "review", "High-cardinality string field"
    return "keep", "No immediate issue"

def save_plot(fig, filename: str):
    path = os.path.join(PLOT_DIR, filename)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path

def downsample_series(series: pd.Series, max_rows: int = PROFILE_SERIES_PLOT_MAX, random_seed: int = RANDOM_SEED) -> pd.Series:
    if len(series) <= max_rows:
        return series
    return series.sample(n=max_rows, random_state=random_seed)


## Read parquet metadata and schema

This section captures the core facts needed for the Description of Data section.

In [ ]:
# 7. Read parquet metadata and schema

append_run_log("Reading parquet metadata and schema...")

parquet_file = pq.ParquetFile(SOURCE_PARQUET)
arrow_schema = parquet_file.schema_arrow
dataset = ds.dataset(SOURCE_PARQUET, format="parquet")

num_row_groups = parquet_file.num_row_groups
num_columns = len(arrow_schema.names)

if USE_METADATA_ROW_COUNT:
    total_rows = parquet_file.metadata.num_rows
else:
    total_rows = None

source_file_size_bytes = os.path.getsize(SOURCE_PARQUET)

print("Schema loaded.")
print("Rows (metadata):", total_rows)
print("Columns:", num_columns)
print("Row groups:", num_row_groups)
print("File size (bytes):", source_file_size_bytes)

append_run_log(
    f"Metadata loaded | rows={total_rows} | columns={num_columns} | row_groups={num_row_groups} | file_size_bytes={source_file_size_bytes}"
)

Schema loaded.
Rows (metadata): 20774822
Columns: 46
Row groups: 104
File size (bytes): 1817114586


In [ ]:
# 8. Create dataset inventory

dataset_inventory = pd.DataFrame([
    {
        "run_id": RUN_ID,
        "run_timestamp": RUN_TS,
        "source_parquet": SOURCE_PARQUET,
        "file_format": "parquet",
        "file_size_bytes": source_file_size_bytes,
        "file_size_mb": round(source_file_size_bytes / (1024 ** 2), 2),
        "row_count_metadata": total_rows,
        "column_count": num_columns,
        "row_group_count": num_row_groups,
        "drive_source": "Google Drive / MyDrive",
        "notebook_stage": "Notebook 1 - Intake and Profiling",
    }
])

dataset_inventory_path = os.path.join(PROFILE_DIR, "dataset_inventory.csv")
dataset_inventory.to_csv(dataset_inventory_path, index=False)

display(dataset_inventory)
append_run_log(f"Saved dataset inventory to {dataset_inventory_path}")

,run_id,run_timestamp,source_parquet,file_format,file_size_bytes,file_size_mb,row_count_metadata,column_count,row_group_count,drive_source,notebook_stage
0,20260422_215232,2026-04-22 21:52:32,/content/drive/MyDrive/project_data/Capstone C...,parquet,1817114586,1732.94,20774822,46,104,Google Drive / MyDrive,Notebook 1 - Intake and Profiling


In [ ]:
# 9. Build initial column profile from schema

column_profile_rows = []
for field in arrow_schema:
    semantic_type = infer_semantic_type(field.type)
    column_profile_rows.append({
        "column_name": field.name,
        "arrow_dtype": str(field.type),
        "semantic_type": semantic_type,
        "nullable": field.nullable,
    })

column_profile = pd.DataFrame(column_profile_rows)
column_profile["column_order"] = np.arange(1, len(column_profile) + 1)

column_profile_path = os.path.join(PROFILE_DIR, "column_profile.csv")
column_profile.to_csv(column_profile_path, index=False)

display(column_profile.head(20))
append_run_log(f"Saved initial schema profile to {column_profile_path}")

,column_name,arrow_dtype,semantic_type,nullable,column_order
0,address_type,string,string,True,1
1,agency,string,string,True,2
2,agency_name,string,string,True,3
3,bbl,string,string,True,4
4,borough,string,string,True,5
5,bridge_highway_direction,string,string,True,6
6,bridge_highway_name,string,string,True,7
7,bridge_highway_segment,string,string,True,8
8,city,string,string,True,9
9,closed_date,string,string,True,10


## Create a development sample

The full dataset is large. This sample is used for:
- quick value inspection
- category exploration
- chart generation
- early field review

The sample does **not** replace the full dataset. It is only for profiling support.

In [ ]:
# 10. Create a development sample from the parquet

append_run_log("Creating development sample...")

rng = np.random.default_rng(RANDOM_SEED)
sample_parts = []
sample_rows_collected = 0

estimated_total_rows = total_rows if total_rows is not None and total_rows > 0 else None
sample_rate = min(1.0, SAMPLE_ROWS_TARGET / estimated_total_rows) if estimated_total_rows else None

scanner = dataset.scanner(batch_size=SAMPLE_BATCH_SIZE)

for batch_number, record_batch in enumerate(scanner.to_batches(), start=1):
    if sample_rows_collected >= SAMPLE_ROWS_TARGET:
        break

    pdf = record_batch.to_pandas()
    if pdf.empty:
        continue

    remaining = SAMPLE_ROWS_TARGET - sample_rows_collected
    if remaining <= 0:
        break

    if sample_rate is not None:
        proposed_n = max(1, int(round(len(pdf) * sample_rate)))
        take_n = min(len(pdf), remaining, proposed_n)
    else:
        take_n = min(len(pdf), remaining)

    if take_n >= len(pdf):
        sampled_batch = pdf
    else:
        sampled_batch = pdf.sample(n=take_n, random_state=RANDOM_SEED + batch_number)

    sample_parts.append(sampled_batch)
    sample_rows_collected += len(sampled_batch)

    if batch_number % 50 == 0:
        append_run_log(f"Sample progress: batches={batch_number} | sample_rows={sample_rows_collected}")

    del pdf
    gc.collect()

sample_df = pd.concat(sample_parts, ignore_index=True) if sample_parts else pd.DataFrame()

if len(sample_df) > SAMPLE_ROWS_TARGET:
    sample_df = sample_df.sample(n=SAMPLE_ROWS_TARGET, random_state=RANDOM_SEED).reset_index(drop=True)

sample_path = os.path.join(PROFILE_DIR, "development_sample.parquet")
sample_df.to_parquet(sample_path, index=False)

print("Development sample shape:", sample_df.shape)
display(sample_df.head())

append_run_log(
    f"Development sample created with shape={sample_df.shape}, batch_size={SAMPLE_BATCH_SIZE}, target_rows={SAMPLE_ROWS_TARGET}, saved_to={sample_path}"
)


Development sample shape: (150000, 46)


,address_type,agency,agency_name,bbl,borough,bridge_highway_direction,bridge_highway_name,bridge_highway_segment,city,closed_date,community_board,complaint_type,council_district,created_date,cross_street_1,cross_street_2,descriptor,descriptor_2,due_date,facility_type,incident_address,incident_zip,intersection_street_1,intersection_street_2,landmark,latitude,location,location_type,longitude,open_data_channel_type,park_borough,park_facility_name,police_precinct,resolution_action_updated_date,resolution_description,road_ramp,source_file,source_parquet,status,street_name,taxi_company_borough,taxi_pick_up_location,unique_key,vehicle_type,x_coordinate_state_plane,y_coordinate_state_plane
0,ADDRESS,HPD,Department of Housing Preservation and Develop...,1018210004.0,MANHATTAN,<NA>,<NA>,<NA>,NEW YORK,<NA>,10 MANHATTAN,ELECTRIC,9.0,2026-04-02T21:25:55.000,<NA>,<NA>,LIGHTING,FIXTURE MISSING/HANGING/LOOSE,<NA>,<NA>,1829 ADAM C POWELL BOULEVARD,10026.0,<NA>,<NA>,<NA>,40.80023112217873,<NA>,RESIDENTIAL BUILDING,-73.95469615818246,PHONE,MANHATTAN,Unspecified,Precinct 28,2026-04-02T00:00:00.000,The following complaint conditions are still o...,<NA>,nyc_311_all_appended.parquet,batch_0.parquet,Open,ADAM C POWELL BOULEVARD,<NA>,<NA>,68539143,<NA>,996793.0,230826.0
1,ADDRESS,DSNY,Department of Sanitation,4035870023.0,QUEENS,<NA>,<NA>,<NA>,RIDGEWOOD,2026-04-03T13:02:05.000,05 QUEENS,Obstruction,34.0,2026-04-03T09:27:28.000,71 AVENUE,60 STREET,Merchandise,Sidewalk,<NA>,<NA>,59-32 MYRTLE AVENUE,11385.0,71 AVENUE,60 STREET,MYRTLE AVENUE,40.700506169955226,<NA>,Sidewalk,-73.89863266106642,ONLINE,QUEENS,Unspecified,Precinct 104,2026-04-03T13:02:07.000,The Department of Sanitation investigated this...,<NA>,nyc_311_all_appended.parquet,batch_0.parquet,Closed,MYRTLE AVENUE,<NA>,<NA>,68556713,<NA>,1012357.0,194506.0
2,ADDRESS,NYPD,New York City Police Department,4156280056.0,QUEENS,<NA>,<NA>,<NA>,FAR ROCKAWAY,2026-04-03T20:11:09.000,14 QUEENS,Illegal Parking,31.0,2026-04-03T19:13:36.000,BEACH 17 STREET,BEACH 19 STREET,Blocked Hydrant,<NA>,<NA>,<NA>,17-08 BROOKHAVEN AVENUE,11691.0,BEACH 17 STREET,BEACH 19 STREET,BROOKHAVEN AVENUE,40.598913344673896,<NA>,Street/Sidewalk,-73.75093345542008,MOBILE,QUEENS,Unspecified,Precinct 101,2026-04-03T20:11:12.000,The New York City Police Department responded ...,<NA>,nyc_311_all_appended.parquet,batch_0.parquet,Closed,BROOKHAVEN AVENUE,<NA>,<NA>,68552976,<NA>,1053416.0,157575.0
3,PLACE,DPR,Department of Parks and Recreation,4162860101.0,QUEENS,<NA>,<NA>,<NA>,FAR ROCKAWAY,2026-04-03T15:58:15.000,14 QUEENS,Animal in a Park,32.0,2026-04-03T15:42:56.000,BEACH 138 STREET,BEACH 139 STREET,Dead Animal,<NA>,<NA>,<NA>,ROCKAWAY BEACH,11694.0,BEACH 138 STREET,BEACH 139 STREET,ROCKAWAY BEACH,40.57081674495128,<NA>,Park,-73.85437947966363,UNKNOWN,QUEENS,Unspecified,Precinct 100,2026-04-03T15:58:18.000,NYC Parks determined that another entity needs...,<NA>,nyc_311_all_appended.parquet,batch_0.parquet,Closed,ROCKAWAY BEACH,<NA>,<NA>,68550834,<NA>,1024706.0,147274.0
4,BLOCKFACE,DOT,Department of Transportation,<NA>,QUEENS,<NA>,<NA>,<NA>,QUEENS,2026-04-03T17:42:21.000,03 QUEENS,Street Condition,<NA>,2026-04-03T17:42:21.000,MOUNT EVEREST WAY,32 AVENUE,Pothole,<NA>,<NA>,<NA>,75 STREET,11370.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,UNKNOWN,QUEENS,Unspecified,Precinct 115,2026-04-03T17:42:21.000,The Department of Transportation determined th...,<NA>,nyc_311_all_appended.parquet,batch_0.parquet,Closed,75 STREET,<NA>,<NA>,68555836,<NA>,<NA>,<NA>


## Full null scan

This scan processes the full parquet in batches to compute null counts by column.  
For a large dataset, this is one of the most important facts for the overview of the data section.

In [ ]:
# 11. Full null scan by column

append_run_log("Starting full null scan...")

null_counts = {col: 0 for col in column_profile["column_name"]}
rows_scanned = 0

if RUN_FULL_NULL_SCAN:
    scanner = dataset.scanner(batch_size=NULL_SCAN_BATCH_SIZE)

    for batch_number, record_batch in enumerate(scanner.to_batches(), start=1):
        rows_scanned += record_batch.num_rows

        for idx, col_name in enumerate(record_batch.schema.names):
            null_counts[col_name] += int(record_batch.column(idx).null_count)

        if batch_number % 100 == 0:
            print(f"Processed {rows_scanned:,} rows so far...")
            append_run_log(f"Null scan progress: rows_scanned={rows_scanned}")

        del record_batch
        gc.collect()
else:
    append_run_log("Full null scan skipped. Null counts will be estimated from sample only.")
    if not sample_df.empty:
        for col in sample_df.columns:
            null_counts[col] = int(sample_df[col].isna().sum())
        rows_scanned = len(sample_df)

null_count_df = pd.DataFrame({
    "column_name": list(null_counts.keys()),
    "null_count": list(null_counts.values()),
})

null_count_df["rows_scanned"] = rows_scanned
null_count_df["null_pct"] = np.where(
    rows_scanned > 0,
    (null_count_df["null_count"] / rows_scanned) * 100,
    np.nan
)

missingness_summary_path = os.path.join(PROFILE_DIR, "missingness_summary.csv")
null_count_df.sort_values("null_pct", ascending=False).to_csv(missingness_summary_path, index=False)

display(null_count_df.sort_values("null_pct", ascending=False).head(20))
append_run_log(f"Saved missingness summary to {missingness_summary_path}")


Processed 4,866,170 rows so far...
Processed 9,866,160 rows so far...
Processed 14,866,158 rows so far...
Processed 19,866,155 rows so far...


,column_name,null_count,rows_scanned,null_pct
26,location,20774822,20774822,100.000000
40,taxi_company_borough,20762990,20774822,99.943046
35,road_ramp,20727250,20774822,99.771011
5,bridge_highway_direction,20704781,20774822,99.662856
18,due_date,20702578,20774822,99.652252
7,bridge_highway_segment,20644835,20774822,99.374305
6,bridge_highway_name,20644546,20774822,99.372914
41,taxi_pick_up_location,20592980,20774822,99.124700
19,facility_type,20549065,20774822,98.913314
43,vehicle_type,20391380,20774822,98.154295


## Sample-based profiling

The next section uses the development sample to collect:
- example values
- estimated distinct counts
- sample min and max
- top category values for selected fields

For early profiling and notebook design.

In [ ]:
# 12. Add sample-based profiling details

append_run_log("Building sample-based column profile details...")

sample_profile_rows = []
top_values_records = []

for col in sample_df.columns:
    series = sample_df[col]
    non_null = series.dropna()
    semantic_type = column_profile.loc[column_profile["column_name"] == col, "semantic_type"].iloc[0]

    sample_non_null_count = int(non_null.shape[0])
    sample_row_count = int(series.shape[0])
    distinct_estimate = int(non_null.nunique(dropna=True)) if sample_non_null_count > 0 else 0

    sample_values = [safe_string(v) for v in non_null.head(MAX_SAMPLE_VALUES_PER_COLUMN).tolist()]

    sample_min = None
    sample_max = None

    if semantic_type in ["numeric", "datetime"] and sample_non_null_count > 0:
        try:
            sample_min = safe_string(non_null.min())
            sample_max = safe_string(non_null.max())
        except Exception:
            pass

    if semantic_type in ["string", "boolean"] and sample_non_null_count > 0:
        top_counts = non_null.astype(str).value_counts(dropna=True).head(MAX_TOP_VALUES_PER_COLUMN)
        for rank, (val, cnt) in enumerate(top_counts.items(), start=1):
            top_values_records.append({
                "column_name": col,
                "rank": rank,
                "value": safe_string(val),
                "count_in_sample": int(cnt),
                "pct_in_sample": round((cnt / sample_row_count) * 100, 4) if sample_row_count > 0 else np.nan
            })

    sample_profile_rows.append({
        "column_name": col,
        "sample_row_count": sample_row_count,
        "sample_non_null_count": sample_non_null_count,
        "sample_distinct_estimate": distinct_estimate,
        "sample_values": " | ".join([v for v in sample_values if v is not None]),
        "sample_min": sample_min,
        "sample_max": sample_max
    })

sample_profile_df = pd.DataFrame(sample_profile_rows)
top_values_df = pd.DataFrame(top_values_records)

distinct_value_summary_path = os.path.join(PROFILE_DIR, "distinct_value_summary.csv")
sample_profile_df[["column_name", "sample_distinct_estimate"]].to_csv(distinct_value_summary_path, index=False)

top_values_path = os.path.join(PROFILE_DIR, "top_values_selected_columns.csv")
top_values_df.to_csv(top_values_path, index=False)

append_run_log(f"Saved distinct value summary to {distinct_value_summary_path}")
append_run_log(f"Saved top values summary to {top_values_path}")

display(sample_profile_df.head(20))

,column_name,sample_row_count,sample_non_null_count,sample_distinct_estimate,sample_values,sample_min,sample_max
0,address_type,150000,129896,6,ADDRESS | ADDRESS | ADDRESS | PLACE | BLOCKFACE,None,None
1,agency,150000,150000,19,HPD | DSNY | NYPD | DPR | DOT,None,None
2,agency_name,150000,150000,19,Department of Housing Preservation and Develop...,None,None
3,bbl,150000,132692,84698,1018210004.0 | 4035870023.0 | 4156280056.0 | 4...,None,None
4,borough,150000,149718,6,MANHATTAN | QUEENS | QUEENS | QUEENS | QUEENS,None,None
5,bridge_highway_direction,150000,508,123,North/Westchester County Bound | East/Long Isl...,None,None
6,bridge_highway_name,150000,948,56,Major Deegan Expwy | F | N | Long Island Expwy...,None,None
7,bridge_highway_segment,150000,946,206,W Fordham Rd Univ Hts Br (Exit 9) | Stairway ...,None,None
8,city,150000,142340,95,NEW YORK | RIDGEWOOD | FAR ROCKAWAY | FAR ROCK...,None,None
9,closed_date,150000,146937,137163,2026-04-03T13:02:05.000 | 2026-04-03T20:11:09....,None,None


In [ ]:
# 13. Merge full and sample profiling into the main column profile

column_profile = column_profile.merge(null_count_df[["column_name", "null_count", "null_pct"]], on="column_name", how="left")
column_profile = column_profile.merge(sample_profile_df, on="column_name", how="left")

column_profile["null_pct"] = column_profile["null_pct"].round(4)

column_profile["recommended_action"] = None
column_profile["reason"] = None

for idx, row in column_profile.iterrows():
    action, reason = classify_recommended_action(
        null_pct=row.get("null_pct", np.nan) if pd.notna(row.get("null_pct", np.nan)) else 0,
        distinct_estimate=row.get("sample_distinct_estimate", 0) if pd.notna(row.get("sample_distinct_estimate", 0)) else 0,
        semantic_type=row.get("semantic_type"),
        sample_non_null_count=row.get("sample_non_null_count", 0) if pd.notna(row.get("sample_non_null_count", 0)) else 0,
        column_name=row.get("column_name")
    )
    column_profile.at[idx, "recommended_action"] = action
    column_profile.at[idx, "reason"] = reason

column_profile.to_csv(column_profile_path, index=False)

display(column_profile.head(25))
append_run_log(f"Updated and saved merged column profile to {column_profile_path}")

,column_name,arrow_dtype,semantic_type,nullable,column_order,null_count,null_pct,sample_row_count,sample_non_null_count,sample_distinct_estimate,sample_values,sample_min,sample_max,recommended_action,reason
0,address_type,string,string,True,1,2790586,13.4325,150000,129896,6,ADDRESS | ADDRESS | ADDRESS | PLACE | BLOCKFACE,None,None,keep,No immediate issue
1,agency,string,string,True,2,0,0.0000,150000,150000,19,HPD | DSNY | NYPD | DPR | DOT,None,None,keep,No immediate issue
2,agency_name,string,string,True,3,0,0.0000,150000,150000,19,Department of Housing Preservation and Develop...,None,None,keep,No immediate issue
3,bbl,string,string,True,4,2413529,11.6176,150000,132692,84698,1018210004.0 | 4035870023.0 | 4156280056.0 | 4...,None,None,review,High-cardinality string field
4,borough,string,string,True,5,38435,0.1850,150000,149718,6,MANHATTAN | QUEENS | QUEENS | QUEENS | QUEENS,None,None,keep,No immediate issue
5,bridge_highway_direction,string,string,True,6,20704781,99.6629,150000,508,123,North/Westchester County Bound | East/Long Isl...,None,None,drop,Very high missingness
6,bridge_highway_name,string,string,True,7,20644546,99.3729,150000,948,56,Major Deegan Expwy | F | N | Long Island Expwy...,None,None,drop,Very high missingness
7,bridge_highway_segment,string,string,True,8,20644835,99.3743,150000,946,206,W Fordham Rd Univ Hts Br (Exit 9) | Stairway ...,None,None,drop,Very high missingness
8,city,string,string,True,9,1067680,5.1393,150000,142340,95,NEW YORK | RIDGEWOOD | FAR ROCKAWAY | FAR ROCK...,None,None,keep,No immediate issue
9,closed_date,string,string,True,10,425462,2.0480,150000,146937,137163,2026-04-03T13:02:05.000 | 2026-04-03T20:11:09....,None,None,review,High-cardinality string field


In [ ]:
# 14. Create candidate drop review file

candidate_drop_review = column_profile[
    [
        "column_name",
        "arrow_dtype",
        "semantic_type",
        "nullable",
        "null_count",
        "null_pct",
        "sample_non_null_count",
        "sample_distinct_estimate",
        "recommended_action",
        "reason",
    ]
].copy()

candidate_drop_review = candidate_drop_review.sort_values(
    by=["recommended_action", "null_pct"], ascending=[True, False]
).reset_index(drop=True)

candidate_drop_review_path = os.path.join(PROFILE_DIR, "candidate_drop_review.csv")
candidate_drop_review.to_csv(candidate_drop_review_path, index=False)

display(candidate_drop_review.head(30))
append_run_log(f"Saved candidate drop review to {candidate_drop_review_path}")

,column_name,arrow_dtype,semantic_type,nullable,null_count,null_pct,sample_non_null_count,sample_distinct_estimate,recommended_action,reason
0,location,string,string,True,20774822,100.0000,0,0,drop,Very high missingness
1,taxi_company_borough,string,string,True,20762990,99.9430,66,5,drop,Very high missingness
2,road_ramp,string,string,True,20727250,99.7710,321,61,drop,Very high missingness
3,bridge_highway_direction,string,string,True,20704781,99.6629,508,123,drop,Very high missingness
4,due_date,string,string,True,20702578,99.6523,537,419,drop,Very high missingness
5,bridge_highway_segment,string,string,True,20644835,99.3743,946,206,drop,Very high missingness
6,bridge_highway_name,string,string,True,20644546,99.3729,948,56,drop,Very high missingness
7,taxi_pick_up_location,string,string,True,20592980,99.1247,1305,1129,drop,Very high missingness
8,facility_type,string,string,True,20549065,98.9133,1624,1,drop,Very high missingness
9,vehicle_type,string,string,True,20391380,98.1543,2736,6,drop,Very high missingness


## Visual profiling

These charts are intended for:
- quick inspection in the notebook
- supporting the overview of the data write-up

In [ ]:
# 15. Missingness bar chart

missing_plot_df = null_count_df.sort_values("null_pct", ascending=False).head(25)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(missing_plot_df["column_name"], missing_plot_df["null_pct"])
ax.set_title("Top 25 Columns by Missing Percentage")
ax.set_xlabel("Missing %")
ax.set_ylabel("Column")
ax.invert_yaxis()

missing_plot_path = save_plot(fig, "missingness_bar_top25.png")
print("Saved:", missing_plot_path)
append_run_log(f"Saved plot {missing_plot_path}")

Saved: /content/project_outputs/01_intake_profile/plots/missingness_bar_top25.png


In [ ]:
# 16. Select columns for additional plots

numeric_plot_cols, categorical_plot_cols, datetime_plot_cols = choose_plot_columns(column_profile)

print("Numeric plot columns:", numeric_plot_cols)
print("Categorical plot columns:", categorical_plot_cols)
print("Datetime plot columns:", datetime_plot_cols)


Numeric plot columns: []
Categorical plot columns: ['address_type', 'agency', 'agency_name', 'bbl', 'borough', 'bridge_highway_direction']
Datetime plot columns: []


In [ ]:
# 17. Numeric distributions

saved_numeric_plots = []

for col in numeric_plot_cols:
    if col not in sample_df.columns:
        continue

    series = pd.to_numeric(sample_df[col], errors="coerce").dropna()
    if series.empty:
        continue

    series = downsample_series(series)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(series, bins=40)
    ax.set_title(f"Distribution of {col} (sample)")
    ax.set_xlabel(col)
    ax.set_ylabel("Frequency")

    filename = f"numeric_distribution_{col}.png".replace("/", "_")
    path = save_plot(fig, filename)
    saved_numeric_plots.append(path)

print("Saved numeric plots:")
for p in saved_numeric_plots:
    print(" -", p)

append_run_log(f"Saved {len(saved_numeric_plots)} numeric distribution plots")


Saved numeric plots:


In [ ]:
# 18. Top categorical value charts

saved_category_plots = []

for col in categorical_plot_cols:
    if col not in sample_df.columns:
        continue

    series = sample_df[col].dropna().astype(str)
    if series.empty:
        continue

    if len(series) > PROFILE_SERIES_PLOT_MAX:
        series = series.sample(n=PROFILE_SERIES_PLOT_MAX, random_state=RANDOM_SEED)

    top_counts = series.value_counts().head(MAX_TOP_VALUES_PER_COLUMN)

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(top_counts.index.astype(str), top_counts.values)
    ax.set_title(f"Top Values for {col} (sample)")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)

    filename = f"top_categories_{col}.png".replace("/", "_")
    path = save_plot(fig, filename)
    saved_category_plots.append(path)

print("Saved categorical plots:")
for p in saved_category_plots:
    print(" -", p)

append_run_log(f"Saved {len(saved_category_plots)} categorical plots")


Saved categorical plots:
 - /content/project_outputs/01_intake_profile/plots/top_categories_address_type.png
 - /content/project_outputs/01_intake_profile/plots/top_categories_agency.png
 - /content/project_outputs/01_intake_profile/plots/top_categories_agency_name.png
 - /content/project_outputs/01_intake_profile/plots/top_categories_bbl.png
 - /content/project_outputs/01_intake_profile/plots/top_categories_borough.png
 - /content/project_outputs/01_intake_profile/plots/top_categories_bridge_highway_direction.png


In [ ]:
# 19. Date coverage plots for detected datetime columns

saved_datetime_plots = []

for col in datetime_plot_cols:
    if col not in sample_df.columns:
        continue

    dt_series = pd.to_datetime(sample_df[col], errors="coerce").dropna()
    if dt_series.empty:
        continue

    dt_series = downsample_series(dt_series)
    month_counts = dt_series.dt.to_period("M").astype(str).value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(month_counts.index, month_counts.values)
    ax.set_title(f"Date Coverage by Month for {col} (sample)")
    ax.set_xlabel("Month")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=45)

    filename = f"date_coverage_{col}.png".replace("/", "_")
    path = save_plot(fig, filename)
    saved_datetime_plots.append(path)

print("Saved datetime plots:")
for p in saved_datetime_plots:
    print(" -", p)

append_run_log(f"Saved {len(saved_datetime_plots)} datetime coverage plots")


Saved datetime plots:


## Save run manifest

Important because it gives Notebook 2 a clean reference to:
- the raw source parquet
- where Notebook 1 outputs were saved
- which handoff files to read next

In [ ]:
# 20. Save run manifest

manifest = {
    "run_id": RUN_ID,
    "run_timestamp": RUN_TS,
    "notebook_stage": "Notebook 1 - Intake and Profiling",
    "source_parquet": SOURCE_PARQUET,
    "output_dir": OUTPUT_DIR,
    "profile_dir": PROFILE_DIR,
    "plot_dir": PLOT_DIR,
    "dataset_inventory_path": dataset_inventory_path,
    "column_profile_path": column_profile_path,
    "missingness_summary_path": missingness_summary_path,
    "distinct_value_summary_path": distinct_value_summary_path,
    "candidate_drop_review_path": candidate_drop_review_path,
    "top_values_selected_columns_path": top_values_path,
    "development_sample_path": sample_path,
    "row_count_metadata": int(total_rows) if total_rows is not None else None,
    "column_count": int(num_columns),
    "full_null_scan_enabled": bool(RUN_FULL_NULL_SCAN),
    "null_scan_batch_size": int(NULL_SCAN_BATCH_SIZE),
    "sample_rows_target": int(SAMPLE_ROWS_TARGET),
    "sample_batch_size": int(SAMPLE_BATCH_SIZE),
    "profile_series_plot_max": int(PROFILE_SERIES_PLOT_MAX),
    "handoff_to_notebook_2": {
        "primary_files": [
            dataset_inventory_path,
            column_profile_path,
            missingness_summary_path,
            candidate_drop_review_path
        ],
        "purpose": "Use these files to define field keep/drop/transform decisions in Notebook 2."
    }
}

manifest_path = os.path.join(OUTPUT_DIR, "run_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print("Saved manifest:", manifest_path)
append_run_log(f"Saved manifest to {manifest_path}")


Saved manifest: /content/project_outputs/run_manifest.json


In [ ]:
# 23. Completion summary

completed_items = [
    "Mounted Google Drive and validated the source parquet path.",
    "Created one local Colab output directory for zip-ready results.",
    "Read parquet metadata and schema.",
    "Saved dataset inventory for the Description of Data section.",
    "Built a column-level profile from the schema and profiling scans.",
    "Computed missingness by column using Arrow batch null counts to reduce RAM pressure.",
    "Created a development sample for visual profiling and value inspection.",
    "Estimated distinct counts and sample values by column from the development sample.",
    "Created a preliminary keep / review / drop file for downstream field decisions.",
    "Generated selected profiling charts from a bounded sample for notebook review and appendix support.",
    "Saved a run manifest so Notebook 2 can read the correct handoff files."
]

print("NOTEBOOK 1 COMPLETION SUMMARY")
print("=" * 70)
for i, item in enumerate(completed_items, start=1):
    print(f"{i}. {item}")

print("\nPrimary files created:")
primary_files = [
    dataset_inventory_path,
    column_profile_path,
    missingness_summary_path,
    candidate_drop_review_path,
    manifest_path
]
for path in primary_files:
    print(" -", path)

print("\nScaling notes for 20M+ rows:")
print(f" - Development sample target: {SAMPLE_ROWS_TARGET:,} rows")
print(f" - Sample batch size: {SAMPLE_BATCH_SIZE:,} rows")
print(f" - Full null scan batch size: {NULL_SCAN_BATCH_SIZE:,} rows")
print(f" - Plot row cap per series: {PROFILE_SERIES_PLOT_MAX:,} rows")

print("\nNext step:")
print("Send the main handoff files and a few key charts so Notebook 2 can be designed from these results.")


NOTEBOOK 1 COMPLETION SUMMARY
1. Mounted Google Drive and validated the source parquet path.
2. Created one local Colab output directory for zip-ready results.
3. Read parquet metadata and schema.
4. Saved dataset inventory for the Description of Data section.
5. Built a column-level profile from the schema and profiling scans.
6. Computed missingness by column using Arrow batch null counts to reduce RAM pressure.
7. Created a development sample for visual profiling and value inspection.
8. Estimated distinct counts and sample values by column from the development sample.
9. Created a preliminary keep / review / drop file for downstream field decisions.
10. Generated selected profiling charts from a bounded sample for notebook review and appendix support.
11. Saved a run manifest so Notebook 2 can read the correct handoff files.

Primary files created:
 - /content/project_outputs/01_intake_profile/dataset_inventory.csv
 - /content/project_outputs/01_intake_profile/column_profile.csv
 -

#------------------End Remove for post ---------------

In [ ]:
# 24. Optional zip cell
# Notebook 1 outputs.

!cd /content && zip -r project_outputs_notebook1.zip project_outputs

  adding: project_outputs/ (stored 0%)
  adding: project_outputs/run_manifest.json (deflated 69%)
  adding: project_outputs/01_intake_profile/ (stored 0%)
  adding: project_outputs/01_intake_profile/distinct_value_summary.csv (deflated 48%)
  adding: project_outputs/01_intake_profile/dataset_inventory.csv (deflated 31%)
  adding: project_outputs/01_intake_profile/development_sample.parquet (deflated 15%)
  adding: project_outputs/01_intake_profile/run_log.txt (deflated 67%)
  adding: project_outputs/01_intake_profile/initial_findings_notes.txt (deflated 67%)
  adding: project_outputs/01_intake_profile/column_profile.csv (deflated 65%)
  adding: project_outputs/01_intake_profile/candidate_drop_review.csv (deflated 72%)
  adding: project_outputs/01_intake_profile/top_values_selected_columns.csv (deflated 68%)
  adding: project_outputs/01_intake_profile/plots/ (stored 0%)
  adding: project_outputs/01_intake_profile/plots/missingness_bar_top25.png (deflated 24%)
  adding: project_outputs/0